# Kafka Demo — Lab 3

### Connect to Kafka Broker Server
Open an SSH tunnel in your terminal and leave it running while you use this notebook.
Replace `<NetID>` with your UIC NetID:
```
ssh -o ServerAliveInterval=60 -L 9092:localhost:9092 <NetID>@cs544-f26.cs.uic.edu -NTf
```

### To kill connection
```
lsof -ti:9092 | xargs kill -9
```

### Setup
```
python -m pip install kafka-python
```

See [bug_list.md](./bug_list.md) for frequent bugs and solutions.

In [3]:
pip install kafka-python

  Using cached kafka_python-3.0.11-py3-none-any.whl.metadata (11 kB)
Using cached kafka_python-3.0.11-py3-none-any.whl (614 kB)

[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
from datetime import datetime
from json import dumps, loads
from time import sleep
from random import randint
from kafka import KafkaConsumer, KafkaProducer

# Update this for your own recitation section :)
topic = 'recitation-x' # replace x with your recitation section

### Producer Mode -> Writes Data to Broker

In [6]:
# Create a producer to write data to kafka
# Ref: https://kafka-python.readthedocs.io/en/master/apidoc/KafkaProducer.html

# [TODO]: Replace '...' with the address of your Kafka bootstrap server
producer = KafkaProducer(bootstrap_servers=["localhost:9092"],
                        value_serializer=lambda x: dumps(x).encode('utf-8'))

# [TODO]: Add cities of your choice
cities = ["Chicago", "Los Angeles", "Dallas", "Minneapolis", "Austin", "New York City"]

# Write data via the producer
print("Writing to Kafka Broker")
for i in range(10):
    data = f'{datetime.now().strftime("%Y-%m-%d %H:%M:%S")},{cities[randint(0,len(cities)-1)]},{randint(18, 32)}ºC'
    print(f"Writing: {data}")
    producer.send(topic=topic, value=data)
    sleep(1)

Writing to Kafka Broker
Writing: 2026-09-25 15:52:30,Los Angeles,29ºC
Writing: 2026-09-25 15:52:31,Dallas,21ºC
Writing: 2026-09-25 15:52:32,Minneapolis,20ºC
Writing: 2026-09-25 15:52:33,Austin,19ºC
Writing: 2026-09-25 15:52:34,Dallas,26ºC
Writing: 2026-09-25 15:52:35,Austin,31ºC
Writing: 2026-09-25 15:52:36,Los Angeles,18ºC
Writing: 2026-09-25 15:52:37,Austin,21ºC
Writing: 2026-09-25 15:52:38,Chicago,25ºC
Writing: 2026-09-25 15:52:39,Dallas,20ºC


### Consumer Mode -> Reads Data from Broker

In [7]:
# Create a consumer to read data from kafka
# Ref: https://kafka-python.readthedocs.io/en/master/apidoc/KafkaConsumer.html

# [TODO]: Complete the missing ... parameters/arguments using the Kafka documentation
consumer = KafkaConsumer(
    "recitation-x",
    bootstrap_servers=["localhost:9092"],
    auto_offset_reset="earliest", #Experiment with different values
    # Commit that an offset has been read
    enable_auto_commit=True,
    # How often to tell Kafka, an offset has been read
    auto_commit_interval_ms=1000
)

print('Reading Kafka Broker')
for message in consumer:
    message = message.value.decode()
    # Default message.value type is bytes!
    print(loads(message))
    os.system(f"echo {message} >> kafka_log.csv")

Reading Kafka Broker
2026-09-23 18:06:18,chicago,30ºC
2026-09-23 18:06:20,chicago,28ºC
2026-09-23 18:06:21,san diego,21ºC
2026-09-23 18:06:22,san diego,28ºC
2026-09-23 18:06:23,san diego,27ºC
2026-09-23 18:06:24,chicago,30ºC
2026-09-23 18:06:25,phoenix,30ºC
2026-09-23 18:06:26,san diego,29ºC
2026-09-23 18:06:27,chicago,21ºC
2026-09-23 18:06:28,san diego,30ºC
2026-09-23 18:29:45,phoenix,31ºC
2026-09-23 18:29:46,phoenix,21ºC
2026-09-23 18:29:47,san diego,26ºC
2026-09-23 18:29:48,chicago,22ºC
2026-09-23 18:29:49,phoenix,23ºC
2026-09-23 18:29:50,san diego,22ºC
2026-09-23 18:29:51,san diego,23ºC
2026-09-23 18:29:52,chicago,22ºC
2026-09-23 18:29:53,san diego,18ºC
2026-09-23 18:29:54,chicago,21ºC
2026-09-23 21:26:21,seattle,23ºC
2026-09-23 21:26:22,portland,21ºC
2026-09-23 21:26:23,chicago,29ºC
2026-09-23 21:26:24,chicago,29ºC
2026-09-23 21:26:25,chicago,23ºC
2026-09-23 21:26:26,seattle,21ºC
2026-09-23 21:26:27,portland,20ºC
2026-09-23 21:26:28,portland,24ºC
2026-09-23 21:26:29,chicago,21ºC
2

KeyboardInterrupt: 

# Use kcat!
It's a CLI (Command Line Interface). Previously known as kafkacat


Ref: https://docs.confluent.io/platform/current/app-development/kafkacat-usage.html

In [21]:
#kcat command: connect to local Kafka broker, specify a topic, and consume messages from the earliest offset
!kcat -b 'localhost:9092' -t 'recitation-c' -C -o 'beginning' -e



"2026-09-25 11:38:58,Pittsburgh,28\u00baC"
"2026-09-25 11:38:59,Pittsburgh,25\u00baC"
"2026-09-25 11:39:00,Chicago,25\u00baC"
"2026-09-25 11:39:01,Pittsburgh,31\u00baC"
"2026-09-25 11:39:02,Pittsburgh,25\u00baC"
"2026-09-25 11:39:03,New York,32\u00baC"
"2026-09-25 11:39:04,New York,21\u00baC"
"2026-09-25 11:39:05,Chicago,32\u00baC"
"2026-09-25 11:39:06,New York,23\u00baC"
"2026-09-25 11:39:07,New York,26\u00baC"
"2026-09-25 15:33:51,Pittsburgh,24\u00baC"
"2026-09-25 15:33:55,New York,18\u00baC"
"2026-09-25 15:33:56,New York,23\u00baC"
"2026-09-25 15:33:57,Chicago,18\u00baC"
"2026-09-25 15:33:58,New York,19\u00baC"
"2026-09-25 15:33:59,Pittsburgh,30\u00baC"
"2026-09-25 15:34:00,New York,27\u00baC"
"2026-09-25 15:34:01,Pittsburgh,30\u00baC"
"2026-09-25 15:34:02,Chicago,21\u00baC"
"2026-09-25 15:34:03,New York,28\u00baC"
% Reached end of topic recitation-c [0] at offset 40: exiting
